# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

---
## 0. Подготовка

In [136]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [137]:
# ============================================================
# Настройка LLM-клиентов
# ============================================================

client = OpenAI(
    base_url="https://polza.ai/api/v1/",
    api_key="pza_4U5FlDhFI2Z21-qHKB1K7pJBWmbwNheJ",
)

MODEL = "deepseek/deepseek-v4-flash"

---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [138]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+ 
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [139]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

In [140]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

✅ Подключение к GitHub MCP установлено!

🔧 Инструменты (26):
-------------------------------------------------------
  • create_or_update_file
    Create or update a single file in a GitHub repository...

  • search_repositories
    Search for GitHub repositories...

  • create_repository
    Create a new GitHub repository in your account...

  • get_file_contents
    Get the contents of a file or directory from a GitHub repository...

  • push_files
    Push multiple files to a GitHub repository in a single commit...

  • create_issue
    Create a new issue in a GitHub repository...

  • create_pull_request
    Create a new pull request in a GitHub repository...

  • fork_repository
    Fork a GitHub repository to your account or specified organization...

  • create_branch
    Create a new branch in a GitHub repository...

  • list_commits
    Get list of commits of a branch in a GitHub repository...

  • list_issues
    List issues in a GitHub repository with filtering options...

 

In [141]:
!npx -v

10.9.2


In [142]:
import shutil
print(shutil.which("npx"))

C:\Program Files\nodejs\npx.CMD


In [143]:
import os
print(os.path.exists(r"C:\Program Files\nodejs\node.exe"))

True


In [144]:
import os, shutil
os.environ["PATH"] += r";C:\Program Files\nodejs"
print(shutil.which("npx"))

C:\Program Files\nodejs\npx.CMD


---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [145]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills"
SKILLS = load_skills(skills_dir)

print(f"Директория skills: {skills_dir}")
print(f"Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"   {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...\n")


Директория skills: e:\практики\hse-agent-system\lectures\lecture_3\hw_3_skills
Загружено skills: 3

   analysis_report: Формирование аналитического отчёта
    Триггер: Этот skill активируется, когда:
- Анализ отдельных промптов (навык `prompt_analy...

   prompt_analysis: Анализ системного промпта
    Триггер: Этот skill активируется, когда нужно:
- Разобрать один системный промпт на соста...

   prompt_generator: Генерация системного промпта
    Триггер: Этот skill активируется, когда:
- Сводный отчёт (навык `analysis_report`) уже сф...



In [146]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills — Навыки агента-аналитика системных промптов

Набор skills, определяющих **как** агент должен действовать при анализе системных промптов, формировании сводных отчётов и генерации новых промптов.

## Список навыков

| Навык | Файл | Краткое описание |
|-------|------|------------------|
| Анализ промпта | [prompt_analysis.md](prompt_analysis.md) | Разбор ОДНОГО системного промпта: метаданные, секции, persona, приёмы prompt engineering, оценка по 6 критериям. Результат сохраняется через `save_analysis` |
| Аналитический отчёт | [analysis_report.md](analysis_report.md) | Агрегация всех сохранённых анализов: матрица приёмов, общие паттерны, лучшие практики, сравнительные оценки. Результат сохраняется через `save_report` |
| Генерация промпта | [prompt_generator.md](prompt_generator.md) | Создание нового системного промпта для заданной предметной области на основе лучших практик из отчёта. Результат сохраняет

In [147]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


print("READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Анализ системного промпта

## Требуемые инструменты
- `list_github_tools` / `load_github_tool` / `call_github_tool` — получение текста промпта из репозитория
- `save_analysis` — сохранение результата

## Когда активировать
Этот skill активируется, когда нужно:
- Разобрать один системный промпт н...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [148]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [149]:
# ============================================================
# Инструменты агента
# ============================================================
import re
# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\U0001f4c2 Директория анализов: {ANALYSES_DIR}")
print(f"\U0001f4c2 Директория отчётов:  {REPORTS_DIR}")


def _rename(name: str) -> str:
    name = (name or "").strip()
    if name.lower().endswith(".md"):
        name = name[:-3]
    name = re.sub(r"[\\/]+", "_", name)
    name = re.sub(r"[^\w\-. ]", "_", name)
    return name 



# ---- save_analysis ----
def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    name = _rename(name)

    path = ANALYSES_DIR / f"{name}.md"
    path.write_text(content, encoding="utf-8")
    return f"Анализ сохранен\nName: {name}, Len: {len(content)}"



# ---- list_analyses ----
def list_analyses() -> str:
    """Получить список всех сохранённых анализов."""
    files = sorted(ANALYSES_DIR.glob('*md'))
    if not files:
        return f"Файлов не обнаружено в {ANALYSES_DIR}"
    lines = [f"Сохраненных анализов {len(files)}"]
    for file in files:
        lines.append(f" - {file.stem} ({file.stat().st_size / 1024:.1f} Кб)")
    return "\n".join(lines)




# ---- read_analysis ----
def read_analysis(name: str) -> str:
    """Прочитать сохранённый анализ промпта по имени."""
    # TODO: прочитать файл ANALYSES_DIR / f"{name}.md"
    # Если файл есть — вернуть "═══ АНАЛИЗ: {name} ═══\n\n{content}"
    # Если нет — вернуть список доступных
    name = _rename(name)
    
    path = ANALYSES_DIR / f"{name}.md"
    if not path.exists():
        return f"Файл {path} не найден"
    content = path.read_text(encoding="utf-8")
    return f"Файл {path}\nКонтент: {content}"




# ---- save_report ----
def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    # TODO: сохранить content в файл REPORTS_DIR / f"{name}.md"
    # Вернуть строку подтверждения
    name = _rename(name)
    path = REPORTS_DIR / f"{name}.md"
    path.write_text(content, encoding="utf-8")
    return f"Файл {name} сохранен, размер {len(content)} "


# ---- read_report ----
def read_report(name: str) -> str:
    """Прочитать сохранённый отчёт."""
    # TODO: прочитать файл REPORTS_DIR / f"{name}.md"
    # Если файл есть — вернуть содержимое
    # Если нет — вернуть список доступных
    name = _rename(name)
        
    path = REPORTS_DIR / f"{name}.md"
    if not path.exists():
        return f"Файл {path} не найден"
    content = path.read_text(encoding="utf-8")
    return f"Файл {path}\nКонтент: {content}"


from pathlib import PurePosixPath

# ---- List files in github repo ----
async def list_repo_files(session) -> list[dict]:
    """
    Обойти репозиторий через MCP и собрать список промпт-файлов.
    Возвращает [{"path": "OpenAI/Codex.md", "name": "Codex.md", "folder": "OpenAI", "size": ...}]
    """
    # Параметры репозитория
    # REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}
    # ...

    REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}
    PROMPT_EXTENSIONS = {".md", ".txt"}
    SKIP_FILES = {"readme.md", "license", "license.md", "contributing.md"}

    files: list[dict] = []
    queue: list[str] = [""]  # "" = корень репозитория

    while queue:
        path = queue.pop(0)
        result = await session.call_tool("get_file_contents", {**REPO, "path": path})
        text = "\n".join(getattr(c, "text", "") for c in result.content)

        if result.isError:
            print(f"\u26a0\ufe0f get_file_contents('{path}'): {text[:200]}")
            continue

        try:
            items = json.loads(text)
        except json.JSONDecodeError:
            print(f"\u26a0\ufe0f '{path}': ответ не JSON: {text[:200]}")
            continue

        if not isinstance(items, list): 
            continue

        for item in items:
            if item.get("type") == "dir":
                queue.append(item["path"])
            elif item.get("type") == "file":
                p = PurePosixPath(item["path"])
                if (
                    p.suffix.lower() in PROMPT_EXTENSIONS
                    and p.name.lower() not in SKIP_FILES
                    and len(p.parts) > 1  
                ):
                    files.append(
                        {
                            "path": item["path"],
                            "name": item["name"],
                            "folder": str(p.parent),
                            "size": item.get("size", 0),
                        }
                    )

    return sorted(files, key=lambda f: f["path"])


📂 Директория анализов: e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\analyses
📂 Директория отчётов:  e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\reports


In [150]:
FILE_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "save_analysis",
            "description": (
                "Сохранить результат анализа ОДНОГО системного промпта в markdown-файл. "
                "Вызывай после завершения анализа по процедуре prompt_analysis."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя файла без расширения"},
                    "content": {"type": "string", "description": "Полный текст анализа в markdown"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_analyses",
            "description": (
                "Получить список всех сохранённых анализов промптов с размерами. "
                "Вызывай перед формированием сводного отчёта."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_analysis",
            "description": "Прочитать сохранённый анализ одного промпта. Имена бери из list_analyses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя файла без расширения"},
                },
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": (
                "Сохранить сводный отчёт или сгенерированный системный промпт в markdown-файл. "
                "Для отчёта используй имя 'analysis_report', для промпта — 'generated_prompt'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя файла без расширения"},
                    "content": {"type": "string", "description": "Полный текст в markdown"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_report",
            "description": (
                "Прочитать сохранённый отчёт. Сводный отчёт называется 'analysis_report'. "
                "Если файла нет — вернётся список доступных."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя файла без расширения"},
                },
                "required": ["name"],
            },
        },
    },
]

LOCAL_TOOLS = {
    "read_skill": read_skill,
    "save_analysis": save_analysis,
    "list_analyses": list_analyses,
    "read_analysis": read_analysis,
    "save_report": save_report,
    "read_report": read_report,
}

print(f"\n\U0001f527 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  \u2022 {name}")


🔧 Локальные инструменты агента (6):
  • read_skill
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report


---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`). 
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [151]:
MCP_META_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "list_github_tools",
            "description": (
                "Показать список доступных инструментов GitHub MCP-сервера с кратким описанием. "
                "Вызывай, когда нужно получить данные из репозитория, но неизвестно, каким инструментом."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "load_github_tool",
            "description": (
                "Загрузить полную схему параметров одного инструмента GitHub MCP. "
                "Вызывай перед первым обращением к нему через call_github_tool."
            ),
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string", "description": "Имя инструмента из list_github_tools"}},
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "call_github_tool",
            "description": (
                "Вызвать инструмент GitHub MCP-сервера. Схему параметров сначала получи "
                "через load_github_tool."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя инструмента"},
                    "arguments": {
                        "type": "string",
                        "description": 'Аргументы в виде JSON-объекта, например {"owner": "asgeirtj", "repo": "system_prompts_leaks", "path": "OpenAI/Codex.md"}',
                    },
                },
                "required": ["name", "arguments"],
            },
        },
    },
]



BASE_SYSTEM_PROMPT = """Ты — агент-аналитик системных промптов.

Твои возможности:
1. SKILLS — процедурные навыки. Ниже список доступных навыков (Level 1).
   Прежде чем выполнять задачу, вызови read_skill и строго следуй загруженной процедуре.
2. GITHUB MCP — доступ к репозиторию с системными промптами.
   Список инструментов: list_github_tools, схема: load_github_tool, вызов: call_github_tool.
3. ФАЙЛЫ — сохранение и чтение результатов:
   save_analysis, list_analyses, read_analysis, save_report, read_report.

Правила:
- Сначала skill, потом действия.
- Не выдумывай содержимое промптов: бери его только из инструментов.
- Задача не выполнена, пока результат не сохранён в файл.
- Закончив, коротко отчитайся: что сделал и в какой файл сохранил.

{skills_metadata}"""

In [152]:
import base64

def make_mcp_tools(session, mcp_tools) -> dict:

    by_name = {tool.name: tool for tool in mcp_tools}

    def list_github_tools() -> str:
        lines = [f"Инструменты GitHub MCP ({len(by_name)}):\n"]
        for name, tool in by_name.items():
            desc = (tool.description or " ").split("\n")[0][:100]
            lines.append(f" - {name}    - {desc}")
        lines.append("\nПолную схему получить через load_github_tool")
        return "\n".join(lines)

    def load_github_tool(name_tool: str) -> str:
        tool = by_name.get(name_tool)
        if tool is None:
            return "Данный инструмент не найден"
        schema = tool.inputSchema or {"type" : "object", "properties" : []}
        return (
            f"Схема : {schema}"
            f"{tool.descrition}\n"
            f"Параметры: \n{json.dumps(schema, ensure_ascii=False, indent=2)}"
        )

    async def call_github_tool(name: str, arguments: str = "{}") -> str:
        if name not in by_name:
            return "Данный инструмент не найден"
        try:
            args = json.loads(arguments) if isinstance(arguments, str) else (arguments or {})
        except json.JSONDecodeError as e:
            return f"Аргументы не разобрались как JSON: {e}. Передай корректный JSON-объект."
        if not isinstance(args, dict):
            return "Аргументы должны быть JSON-объектом."
        try:
            result = await session.call_tool(name, args)
        except Exception as e:
            return f"Ошибка вызова '{name}': {type(e).__name__}: {e}"

        text = "\n".join(getattr(c, "text", "") for c in result.content)
        if result.isError:
            return f"{name}: {text[:500]}"
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            return text
        if isinstance(payload, dict) and payload.get("encoding") == "base64" and payload.get("content"):
            try:
                return base64.b64decode(payload["content"]).decode("utf-8", errors="replace")
            except Exception:
                return text
        return text


    
    return {
            "list_github_tools": list_github_tools,
            "load_github_tool": load_github_tool,
            "call_github_tool": call_github_tool,
        }
    
    




In [153]:
# ============================================================
# Универсальный агент с tool loop
# ============================================================

# ⚠️ Этап анализа всех промптов может быть долгим, т.к. для каждого файла
# выполняется отдельный агентный вызов с несколькими шагами tool loop.
# Для ускорения вместо синхронных последовательных запросов можно создать
# N потоков (asyncio.gather / ThreadPoolExecutor) и запустить анализ
# нескольких промптов параллельно.
# Как это реализовать — идете на поклон к @Rinaaaa_chan

import inspect
import functools

async def run_agent(
    question: str,
    session, 
    mcp_tools,
    skill_metadata: str = "",
    extra_system: str = "",
    max_steps: int = 10,
    verbose: bool = True,
    temperature: float = 0.2,
    model: str = "deepseek/deepseek-v4-flash"
) -> AgentTrace:
    # ...
    # pass

    meta_tools = make_mcp_tools(session, mcp_tools)
    tool_functions = {**LOCAL_TOOLS, **meta_tools}
    tools_schema = [*FILE_TOOL_SCHEMAS, *MCP_META_TOOL_SCHEMAS, READ_SKILL_TOOL_SCHEMA]
    prev_context = 0


    system_prompt = BASE_SYSTEM_PROMPT.format(skills_metadata=skills_metadata)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user" ,"content": question}
    ]
    trace = AgentTrace(question=question)

    if verbose:
        print(f" Задача {question}")

    for step_num in range(1, max_steps + 1):
        if verbose:
            print(f" ---Шаг {step_num}")

        response = await asyncio.to_thread(
            client.chat.completions.create,
            model=model, 
            tools=tools_schema,
            temperature=temperature,
            messages=messages
            )

        msg = response.choices[0].message
        messages.append(msg)

        usage = _extract_usage(response)
        step = AgentStep(
            step_number=step_num,
            prompt_tokens=usage["prompt_tokens"],
            cached_tokens=usage["cached_tokens"],
            context_size=usage["context_size"],
            context_size_delta=usage["context_size"] - prev_context,
            completion_tokens=usage["completion_tokens"],
            total_tokens=usage["total_tokens"],
        )

        prev_context = usage["context_size"]

        trace.prompt_tokens_total += usage["prompt_tokens"]
        trace.completion_tokens_total += usage["completion_tokens"]
        trace.latest_context_size = usage["context_size"]
        trace.max_context_size = max(trace.max_context_size, usage["context_size"])
        trace.total_tokens_total += usage["total_tokens"]

        if verbose:
            print(
                f" Контекст = {step.context_size}"
                f" complection = {step.completion_tokens}"
            )


        if not msg.tool_calls:
            step.final_answer = msg.content
            trace.steps.append(step)
            trace.final_answer = msg.content
            trace.total_steps = step_num
            if verbose:
                print(f"{(msg.content or '')[:300]}")
            break

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or {})
            except json.JSONDecodeError as e:
                print(f"Ошибка {e}")
                args = {}
            step.tool_calls.append({"name" : name, "args": args})
            if verbose:
                print(f"{name}({json.dumps(args, ensure_ascii=False)[:120]})")

            func = tool_functions.get(name)
            try:
                if inspect.iscoroutinefunction(func):
                    observation = await func(**args)
                else:
                    observation = await asyncio.to_thread(functools.partial(func, **args))
            except Exception as e:
                print(f"Ошибка {e}")
            except TypeError as te:
                print(f"Неверные аргументы {te}")

            observation = str(observation)

            step.observations.append(observation)
            if verbose:
                print(f"{observation[:150]}{'...' if len(observation) > 150 else ''}")
                
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": observation
            })         

        trace.steps.append(step)

    return trace

In [155]:
async def analyze_one_prompt(session, mcp_tools, file_info: dict) -> AgentTrace:
    """Субагент агентный вызов на один фАЙЛ"""
    target_name = _rename(file_info['path'])
    question = (
        f"Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.\n"
        f"Путь к файлу: {file_info['path']}\n"
        f"Сохрани результат через save_analysis с именем '{target_name}'."
    )
    return await run_agent(question=question, session=session, mcp_tools=mcp_tools,
                           skill_metadata=skills_metadata, max_steps=8, verbose=True)

In [154]:
CONCURRENCY = 4

In [ ]:
async def analyze_all_prompts(session, mcp_tools, files: list[dict]) -> list[AgentTrace]:
    semaphore = asyncio.Semaphore(CONCURRENCY)      

    async def guarded(file_info):
        async with semaphore:
            try:
                return await analyze_one_prompt(session, mcp_tools, file_info)
            except Exception as ex:                 
                failed = AgentTrace(question=file_info["path"])
                failed.final_answer = f"Ошибка: {ex}"
                return failed

    return await asyncio.gather(*(guarded(f) for f in files))

In [160]:
# ============================================================
# Тест кейс 1. Запрос на начало анализа 
# ============================================================


In [157]:
async def stage_analysis(session, mcp_tools):
    files = await list_repo_files(session)         
    traces = await analyze_all_prompts(session, mcp_tools, files)
    return traces, files

In [ ]:
async def with_github(work):
    """
    Открыть MCP-сессию 
    """
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools = (await session.list_tools()).tools
                return await work(session, tools)

In [179]:
analiz_traces, analiz_files = asyncio.get_event_loop().run_until_complete(with_github(stage_analysis))
print(f"\n{'=' * 50}")
print(f"Субагентов отработало: {len(analiz_traces)}")
print(f"Суммарно токенов: {sum(t.total_tokens_total for t in analiz_traces)}")
print(f"Максимальный контекст одного субагента: {max((t.max_context_size for t in analiz_traces), default=0)}")
print(f"\n{list_analyses()}")


if analiz_traces:
    print(f"\n{'=' * 50}\nТрассировка первого субагента:\n")
    print(analiz_traces[0].summary())


 Задача Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.
Путь к файлу: Anthropic/anthropic-interviewer.md
Сохрани результат через save_analysis с именем 'Anthropic_anthropic-interviewer'.
 ---Шаг 1
 Задача Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.
Путь к файлу: Anthropic/anthropic_reminders.md
Сохрани результат через save_analysis с именем 'Anthropic_anthropic_reminders'.
 ---Шаг 1
 Задача Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.
Путь к файлу: Anthropic/claude-code/agents/Explore.md
Сохрани результат через save_analysis с именем 'Anthropic_claude-code_agents_Explore'.
 ---Шаг 1
 Задача Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.
Путь к файлу: Anthropic/claude-code/agents/Plan.md
Сохрани результат через save_analysis с именем 'Anthropic_claude-code_agents_Plan'.
 ---Шаг 1
 Контекст = 2004 complection = 108
read_skill({"name": "prompt_analysis"})
═══ SKILL З

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\solov\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000029E158C2D40> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\solov\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000029E158C2D40> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\solov\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self.

 Контекст = 96899 complection = 2690
save_analysis({"name": "Anthropic_claude-code_claude-code-fable-5", "content": "# Анализ: Anthropic / Claude Code (Fable 5)\n\n## Мета)
Анализ сохранен
Name: Anthropic_claude-code_claude-code-fable-5, Len: 4896
 ---Шаг 6
 Контекст = 191576 complection = 296
Анализ выполнен и сохранён. Вот краткий отчёт:

**Что сделано:**
- Загружен навык `prompt_analysis` и строго соблюдена его процедура
- Получен текст системного промпта из репозитория `asgeirtj/system_prompts_leaks` по пути `Anthropic/claude-code/claude-code-fable-5.md` (328 167 символов)
- Выделены 

Субагентов отработало: 412
Суммарно токенов: 1662636
Максимальный контекст одного субагента: 193061

Сохраненных анализов 26
 - Anthropic_anthropic-interviewer (5.4 Кб)
 - Anthropic_anthropic_reminders (6.7 Кб)
 - Anthropic_claude-code_agents_claude-code-guide (6.9 Кб)
 - Anthropic_claude-code_agents_claude (6.0 Кб)
 - Anthropic_claude-code_agents_Explore (6.1 Кб)
 - Anthropic_claude-code_agents_gene

In [ ]:
if analiz_traces:
    print(f"\n{'=' * 50}\nТрассировка первого субагента:\n")
    print(analiz_traces[0].summary())


Трассировка первого субагента:

Вопрос: Проанализируй системный промпт из репозитория asgeirtj/system_prompts_leaks.
Путь к файлу: Anthropic/anthropic-interviewer.md
Сохрани результат через save_analysis с именем 'Anthropic_anthropic-interviewer'.
Шагов: 7
Prompt tokens (биллинг): 36922
Completion tokens: 3779
Всего tokens: 40701

--- Шаг 1 ---
  🔢 context=2002 (Δ +2002), completion=138
  🔧 read_skill({"name": "prompt_analysis"})
  📡 ═══ SKILL ЗАГРУЖЕН: Анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Анализ системног...
--- Шаг 2 ---
  🔢 context=5306 (Δ +3304), completion=133
  🔧 list_github_tools({})
  📡 Инструменты GitHub MCP (26):

 - create_or_update_file    - Create or update a single file in a GitHub repository
 - sea...
--- Шаг 3 ---
  🔢 context=7378 (Δ +2072), completion=66
  🔧 load_github_tool({"name": "get_file_contents"})
  📡 Инструменты GitHub MCP (26):

 - create_or_update_file    - Create or update a single file in a GitHub repository


In [161]:
# ============================================================
# Тест кейс 2. Запрос на формирование итоговой сводки
# ============================================================

In [183]:
async def stage_report(session, mcp_tools):
    return await run_agent(
        question=(
                    "Сформируй сводный аналитический отчёт по всем сохранённым анализам "
                    "системных промптов и сохрани его как 'analysis_report'."
                ),
                session=session,
                mcp_tools=mcp_tools,
                skill_metadata=skills_metadata,
                max_steps=14,
                verbose=True,
    )


In [184]:
report_trace = asyncio.get_event_loop().run_until_complete(with_github(stage_report))

print(
    f"\n{'_ ' * 50 }"
    f"{report_trace.summary()}"
    f"\n{'_ ' * 50 }"
    f"{read_report("analisis_report")}"
    )

 Задача Сформируй сводный аналитический отчёт по всем сохранённым анализам системных промптов и сохрани его как 'analysis_report'.
 ---Шаг 1
 Контекст = 1972 complection = 104
read_skill({"name": "analysis_report"})
═══ SKILL ЗАГРУЖЕН: Формирование аналитического отчёта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Формирование аналитического отчёта

#...
 ---Шаг 2
 Контекст = 4846 complection = 49
list_analyses({})
Сохраненных анализов 26
 - Anthropic_anthropic-interviewer (5.4 Кб)
 - Anthropic_anthropic_reminders (6.7 Кб)
 - Anthropic_claude-code_agents_claude-c...
 ---Шаг 3
 Контекст = 6420 complection = 267
read_analysis({"name": "Anthropic_anthropic-interviewer"})
Файл e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\analyses\Anthropic_anthropic-interviewer.md
Контент: # Анализ: Anthropic / anthropic-...
read_analysis({"name": "Anthropic_anthropic_reminders"})
Файл e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\analyses\Anthropic_anthropic_remin

In [ ]:
# ============================================================
# Тест кейс 3. Запрос на формирование промпта для случайного ассистента
# ============================================================

In [185]:
TARGET_ASSISTENT = (
    ""
)

In [ ]:
async def stage_generator(session, mcp_tools):
    return await run_agent(
        question=(f"Сгенерируй системный промпт для нового ассистента. Описание: {TARGET_ASSISTENT}."
                  "Опирайся на лучшие практики из сохраненного отчета analysis_report.md"
                  "Сохрани результат как 'generated_prompt'"),
        session=session,
        mcp_tools=mcp_tools,
        skill_metadata=skills_metadata,
        max_steps=10,
        verbose=True,
    )


In [189]:
generator_trace = asyncio.get_event_loop().run_until_complete(with_github(stage_generator))

print(f"\n{'=' * 50}")
print(generator_trace.summary())
print(f"\n{'=' * 50}\nСгенерированный промпт:\n")
print(read_report("generated_prompt"))

 Задача Сгенерируй системный промпт для нового ассистента. Описание: .Опирайся на лучшие практики из сохраненного отчета analysis_report.mdСохрани результат как 'generated_prompt
 ---Шаг 1
 Контекст = 1989 complection = 128
read_skill({"name": "prompt_generator"})
═══ SKILL ЗАГРУЖЕН: Генерация системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Генерация системного промпта

## Требуемые ...
 ---Шаг 2
 Контекст = 4878 complection = 67
read_report({"name": "analysis_report"})
Файл e:\практики\hse-agent-system\lectures\lecture_3\hw_3_output\reports\analysis_report.md
Контент: # Сводный отчёт по системным промптам

## Обзор
П...
 ---Шаг 3
 Контекст = 10201 complection = 4815
save_report({"name": "generated_prompt", "content": "# Системный промпт: DevAssist — ассистент-разработчик\n\n## Допущения\n- **Пред)
Файл generated_prompt сохранен, размер 12195 
 ---Шаг 4
 Контекст = 18049 complection = 461
## Готово! Промпт сгенерирован и сохранён

### Что сделано

На основе

<img src="pictures\hw_file_1.PNG" width="600"/>

<img src="pictures\hw_file_2.PNG" width="600"/>